# 역전파를 손으로 (2-2-1)

규칙을 정하지 않으면 shape 이 꼬인다. 이 노트의 규칙(원문 3-5 필수 유도 6항 + 편향 2항):

| # | 수식 | 코드 | shape |
|---|---|---|---|
| 순 | z₁ = W₁x + b₁ | `z1 = W1 @ x + b1` | (2,) |
| 순 | a₁ = σ(z₁) | `a1 = 1/(1+exp(-z1))` | (2,) |
| 순 | z₂ = W₂ᵗa₁ + b₂ | `z2 = W2 @ a1 + b2` | () |
| 순 | ŷ = σ(z₂) | `yhat = 1/(1+exp(-z2))` | () |
| 순 | L = −[y log ŷ + (1−y) log(1−ŷ)] | `bce(yhat, y)` | () |
| **1** | ∂L/∂y_pred = −y/ŷ + (1−y)/(1−ŷ) | `dy_pred = -(y/yhat) + (1-y)/(1-yhat)` | () |
| **2** | ∂L/∂z₂ = ∂L/∂y_pred · ŷ(1−ŷ) = ŷ − y | `dz2 = dy_pred * yhat*(1-yhat)` | () |
| **3** | ∂L/∂W₂ = a₁ · ∂L/∂z₂ | `dW2 = a1 * dz2` | (2,) |
| **4** | ∂L/∂a₁ = W₂ · ∂L/∂z₂ | `da1 = W2 * dz2` | (2,) |
| **5** | ∂L/∂z₁ = ∂L/∂a₁ ⊙ a₁(1−a₁) | `dz1 = da1 * a1*(1-a1)` | (2,) |
| **6** | ∂L/∂W₁ = ∂L/∂z₁ xᵗ | `dW1 = np.outer(dz1, x)` | (2,2) |
| + | ∂L/∂b₂ = ∂L/∂z₂ | `db2 = dz2` | () |
| + | ∂L/∂b₁ = ∂L/∂z₁ | `db1 = dz1` | (2,) |

**1~6 이 원문이 요구한 필수 유도다.** 2번은 1번을 거쳐야 나온다(시그모이드와 BCE 가 상쇄해 ŷ−y 가 되는 것이지, 처음부터 ŷ−y 가 정의가 아니다). 이전 버전 노트는 1·4번을 생략하고 2번부터 시작했다 — 감사에서 지적돼 복원했다.

In [1]:
import numpy as np, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
import backprop as B
x, W1, b1, W2, b2 = B.X, B.W1, B.B1, B.W2, B.B2
print('W1', W1.shape, 'x', x.shape, 'b1', b1.shape, 'W2', W2.shape, 'b2', np.shape(b2))

W1 (2, 2) x (2,) b1 (2,) W2 (2,) b2 ()


## 1. 순전파: 숫자를 하나씩 (원문 3-4 중간값 z1, a1, z2, y_pred 기록)

In [2]:
z1, a1, z2, yhat = B.forward()
print('z1 =', z1, '\na1 =', a1, '\nz2 =', round(z2,6), '\ny_pred = ŷ =', round(yhat,6))

z1 = [0.1 0.3] 
a1 = [0.52497919 0.57444252] 
z2 = 0.607155 
y_pred = ŷ = 0.647292


## 2. 손으로 확인 (σ(0.1)=0.524979 …)

z₁ = W₁x+b₁ = [0.1, 0.3]
a₁ = σ(z₁) = [0.524979, 0.574443]
z₂ = 0.5·0.524979 + 0.6·0.574443 = 0.607155
ŷ = σ(0.607155) = 0.647292
L(y=1) = −log 0.647292 = 0.434958

In [3]:
print('손계산 대조:', abs(round(float(yhat),6) - 0.647292) < 1e-6, abs(round(float(B.bce(yhat,1.0)),6) - 0.434958) < 1e-6)

손계산 대조: True True


## 3. 역전파 6항 + 편향 — 손값과 코드를 나란히

검산은 **같은 수식을 뒤로 돌리는** 방식이 아니라 손실 함수를 흔드는 방식이어야 항등식이 되지 않는다.

In [4]:
for y in (1.0, 0.0):
    h, n = B.backward(y), B.numeric_grad(y)
    print('=== y_true =', y, '===')
    print(' L           =', round(h['L'], 6))
    print(' ∂L/∂y_pred  =', round(h['dy_pred'], 6), '  손: -(y/ŷ)+(1-y)/(1-ŷ) =', round(-(y/0.647292)+(1-y)/(1-0.647292), 6))
    print(' ∂L/∂z₂      =', round(float(h['dz2']), 6), ' = ∂L/∂y_pred·ŷ(1-ŷ) =', round(h['dy_pred']*0.647292*(1-0.647292), 6), ' = ŷ-y =', round(0.647292-y, 6))
    print(' ∂L/∂W₂      =', np.round(h['dW2'], 6), ' shape', h['dW2'].shape)
    print(' ∂L/∂a₁      =', np.round(h['da1'], 6), ' shape', h['da1'].shape, '  손: W₂·∂L/∂z₂ =', np.round(np.array([0.5,0.6])*float(h['dz2']), 6))
    print(' ∂L/∂z₁      =', np.round(h['dz1'], 6), ' shape', h['dz1'].shape)
    print(' ∂L/∂W₁      =', np.round(h['dW1'], 6), ' shape', h['dW1'].shape)
    print(' ∂L/∂b₁      =', np.round(h['db1'], 6), ' / ∂L/∂b₂ =', round(float(h['db2']), 6))
    for k in ('dW1','dW2','db1','db2'):
        e = B.grad_errs(h[k], n[k])
        print(f'   gradcheck {k}: 상대오차(0 제외) {e["상대오차(0이_아닌_항)"]:.3e} · 절대오차 {e["절대오차(전체)"]:.3e}')

=== y_true = 1.0 ===
 L           = 0.434958
 ∂L/∂y_pred  = -1.544899   손: -(y/ŷ)+(1-y)/(1-ŷ) = -1.544898
 ∂L/∂z₂      = -0.352708  = ∂L/∂y_pred·ŷ(1-ŷ) = -0.352708  = ŷ-y = -0.352708
 ∂L/∂W₂      = [-0.185165 -0.202611]  shape (2,)
 ∂L/∂a₁      = [-0.176354 -0.211625]  shape (2,)   손: W₂·∂L/∂z₂ = [-0.176354 -0.211625]
 ∂L/∂z₁      = [-0.043979 -0.051734]  shape (2,)
 ∂L/∂W₁      = [[-0.043979 -0.      ]
 [-0.051734 -0.      ]]  shape (2, 2)
 ∂L/∂b₁      = [-0.043979 -0.051734]  / ∂L/∂b₂ = -0.352708
   gradcheck dW1: 상대오차(0 제외) 2.041e-10 · 절대오차 1.056e-11
   gradcheck dW2: 상대오차(0 제외) 1.609e-11 · 절대오차 3.260e-12
   gradcheck db1: 상대오차(0 제외) 2.041e-10 · 절대오차 1.056e-11
   gradcheck db2: 상대오차(0 제외) 2.035e-11 · 절대오차 7.178e-12
=== y_true = 0.0 ===
 L           = 1.042114
 ∂L/∂y_pred  = 2.835203   손: -(y/ŷ)+(1-y)/(1-ŷ) = 2.835206
 ∂L/∂z₂      = 0.647292  = ∂L/∂y_pred·ŷ(1-ŷ) = 0.647291  = ŷ-y = 0.647292
 ∂L/∂W₂      = [0.339815 0.371832]  shape (2,)
 ∂L/∂a₁      = [0.323646 0.388375]  shape (2,) 

## 4. 왜 ∂L/∂W₁ 의 두 번째 열이 0인가

x=[1,0] 이라 W₁ 의 두 번째 열은 순전파에 참여하지 않는다. **기울기 0 = 오동작이 아니라 입력 결핍.**
그래서 검사는 '모든 항이 0이 아니어야 한다'가 아니라 '0이 아닌 항이 존재해야 한다'여야 한다.
같은 이유로 ∂L/∂a₁ 의 두 성분은 둘 다 살아 있다(W₂=[0.5,0.6] 이 0이 아니므로).

In [5]:
dW1 = B.backward(1.0)['dW1']
print('0이 아닌 항 개수:', int(np.count_nonzero(dW1)), '/', dW1.size)
print('da1 0이 아닌 항:', int(np.count_nonzero(B.backward(1.0)['da1'])), '/', B.backward(1.0)['da1'].size)

0이 아닌 항 개수: 2 / 4
da1 0이 아닌 항: 2 / 2


## 5. y_true 가 미션에 없다 (원문 결함)

정의서가 비어 있어 **양쪽**을 다 실었다. 리포트는 y=1.0 을 표준으로 쓰되, 이 표를 함께 인용한다.

In [6]:
import json, pathlib
d = json.loads(pathlib.Path('reports/backprop.json').read_text(encoding='utf-8'))
print('원문 3-5 필수유도 표:'); print(json.dumps(d['원문3-5_필수유도']['표'], ensure_ascii=False, indent=1))
print('y_true 별 6항 값:'); print(json.dumps(d['원문3-5_필수유도']['y_true_별_값'], ensure_ascii=False, indent=1))
print('게이트:', json.dumps({k: {kk: v[kk] for kk in ('L','dz2','dW2','dW1','최대상대오차_0제외')} for k, v in d['y_true_별'].items()},
                 ensure_ascii=False, indent=1))

원문 3-5 필수유도 표:
[
 {
  "항": "∂L/∂y_pred",
  "키": "dy_pred",
  "shape": "()",
  "수식": "-(y/ŷ) + (1-y)/(1-ŷ)"
 },
 {
  "항": "∂L/∂z2",
  "키": "dz2",
  "shape": "()",
  "수식": "∂L/∂y_pred · ŷ(1-ŷ) = ŷ - y"
 },
 {
  "항": "∂L/∂W2",
  "키": "dW2",
  "shape": "(2,)",
  "수식": "a1 · dz2"
 },
 {
  "항": "∂L/∂a1",
  "키": "da1",
  "shape": "(2,)",
  "수식": "W2 · dz2"
 },
 {
  "항": "∂L/∂z1",
  "키": "dz1",
  "shape": "(2,)",
  "수식": "∂L/∂a1 ⊙ a1(1-a1)"
 },
 {
  "항": "∂L/∂W1",
  "키": "dW1",
  "shape": "(2,2)",
  "수식": "outer(dz1, x)"
 }
]
y_true 별 6항 값:
{
 "1.0": {
  "dy_pred": -1.544899,
  "dz2": -0.352708,
  "dW2": [
   -0.185165,
   -0.202611
  ],
  "da1": [
   -0.176354,
   -0.211625
  ],
  "dz1": [
   -0.043979,
   -0.051734
  ],
  "dW1": [
   -0.043979,
   -0.0,
   -0.051734,
   -0.0
  ]
 },
 "0.0": {
  "dy_pred": 2.835203,
  "dz2": 0.647292,
  "dW2": [
   0.339815,
   0.371832
  ],
  "da1": [
   0.323646,
   0.388375
  ],
  "dz1": [
   0.08071,
   0.094941
  ],
  "dW1": [
   0.08071,
   0.0,
   0.09